<a href="https://colab.research.google.com/github/sukhadaghewari/tropicalcyclone-ri-bench/blob/main/notebooks/01_prepare_storm_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Input data: storm tracks, features and RI labels

## Setup

The next cell works out the repo folder and makes the code in `src/` importable, so the notebook runs the same way locally and in Colab. In Colab it first clones this repo and moves into it.

In [ ]:
# Setup: in Colab, clone the repo and move into it. Then put src/ on the import path.
import os, subprocess, sys
from pathlib import Path

REPO = "tropicalcyclone-ri-bench"
if "google.colab" in sys.modules:
    try:
        os.getcwd()
    except FileNotFoundError:              # the folder we were in was deleted (e.g. an old clone): start again
        os.chdir("/content")
if "google.colab" in sys.modules and os.path.basename(os.getcwd()) != REPO:
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", f"https://github.com/sukhadaghewari/{REPO}.git"], check=True)
    os.chdir(REPO)
if "google.colab" in sys.modules:
    # A clone from an earlier run survives "Restart session": bring it up to date with GitHub.
    if subprocess.run(["git", "pull", "--ff-only"]).returncode != 0:
        print("WARNING: could not update the repo with git pull; using the files as they are.")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # repo root, locally or in Colab
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import numpy as np
import requests
import pandas as pd
from pandas.api.types import is_datetime64_any_dtype
pd.set_option('display.max_columns', None)

In [47]:
# @title Constants used in the code

# ---------------------------------------------------------
# Source data
# ---------------------------------------------------------

IBTRACS_AFTER_1980_CSV_URL = (
    "https://www.ncei.noaa.gov/data/"
    "international-best-track-archive-for-climate-stewardship-ibtracs/"
    "v04r01/access/csv/ibtracs.since1980.list.v04r01.csv"
)

# ---------------------------------------------------------
# GitHub release
# ---------------------------------------------------------

GITHUB_RELEASE_API_URL = (
    "https://api.github.com/repos/"
    "ccai-tropicalcycloneri-tutorial-2026/"
    "tropicalcycloneriprediction/releases/tags/dataset_v4"
)

response = requests.get(GITHUB_RELEASE_API_URL, timeout=60)
response.raise_for_status()          # e.g. GitHub API rate limit: fail here with a clear message
release = response.json()

ASSET_URLS = {
    asset["name"]: asset["browser_download_url"]
    for asset in release["assets"]
}

ERA5_FEATURES_URL = ASSET_URLS["era5_features.parquet"]
SATELLITE_FEATURES_URL = ASSET_URLS["satellite_features.parquet"]

ERA5_FEATURES_FILE = "era5_features.parquet"
SATELLITE_FEATURES_FILE = "satellite_features.parquet"

# ---------------------------------------------------------
# Storm categories
# ---------------------------------------------------------

STORM_CATEGORIES = [
    (0, "Tropical Depression"),
    (34, "Tropical Storm"),
    (64, "Category 1"),
    (83, "Category 2"),
    (96, "Category 3"),
    (113, "Category 4"),
    (137, "Category 5"),
]

# ---------------------------------------------------------
# Basin information
# ---------------------------------------------------------

BASIN_NAMES = {
    "NA": "North Atlantic",
    "EP": "Eastern North Pacific",
    "WP": "Western North Pacific",
    "NI": "North Indian",
    "SI": "South Indian",
    "SP": "Southern Pacific",
    "SA": "South Atlantic",
}

# ---------------------------------------------------------
# Data columns
# ---------------------------------------------------------

NUMERIC_IBTRACS_COLUMNS = [
    "season",
    "lat",
    "lon",
    "usa_wind",
    "dist2land",
    "storm_speed",
]

# ---------------------------------------------------------
# RI definition
# ---------------------------------------------------------

RI_THRESHOLD_KT = 30
RI_HOURS_IN_FUTURE = 24

# ---------------------------------------------------------
# Test data (after 2016)
# ---------------------------------------------------------

TEST_START = 2016
TEST_END = 2025


In [48]:
df_ibtracs = pd.read_csv(
    IBTRACS_AFTER_1980_CSV_URL,
    delimiter=",",
    header=0,
    skiprows=[1],
    keep_default_na=False,
    low_memory=False,
)

print(f"raw ibtracs shape: {df_ibtracs.shape}")

raw ibtracs shape: (310836, 174)


In [49]:
# @title Clean the data and type coercion

# 1. Lower all column names
df_ibtracs.columns = df_ibtracs.columns.str.lower().str.strip()

# 2. Strip whitespace from all text columns at once
text_columns = df_ibtracs.select_dtypes(include=["object", "string"]).columns
df_ibtracs[text_columns] = df_ibtracs[text_columns].apply(lambda s: s.str.strip())

# 3. Convert observation time to datetime
df_ibtracs["iso_time"] = pd.to_datetime(df_ibtracs["iso_time"], errors="coerce")

# 4. Process numeric columns
numeric_cols = [c for c in NUMERIC_IBTRACS_COLUMNS if c in df_ibtracs.columns]
df_ibtracs[numeric_cols] = df_ibtracs[numeric_cols].apply(pd.to_numeric, errors="coerce")

# 5. Longitude: convert -180–180 to 0–360 for easy processing
df_ibtracs["lon"] = df_ibtracs["lon"] % 360

# 6. Add basin metadata
df_ibtracs["full_basin_name"] = df_ibtracs["basin"].map(BASIN_NAMES)

df_ibtracs = df_ibtracs.copy()

/tmp/ipykernel_1287/3423593918.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ibtracs["full_basin_name"] = df_ibtracs["basin"].map(BASIN_NAMES)


In [50]:
# @title Filter the data

MASK = (
    (df_ibtracs["track_type"].str.lower() == "main")
    & (df_ibtracs["nature"].astype(str).str.upper().str.strip() == "TS")
    & (df_ibtracs["iso_time"].dt.hour.isin([0, 6, 12, 18]))
    & (df_ibtracs["iso_time"].dt.minute == 0)
    & (df_ibtracs["usa_wind"].notna())
    & (df_ibtracs["usa_wind"] > 0)
    & (df_ibtracs["dist2land"] > 0)
    & (df_ibtracs["iflag"].str[0].isin(["O", "V"]))
    & (df_ibtracs["season"] <= 2025)
)

df_ibtracs = df_ibtracs[MASK]
df_ibtracs = df_ibtracs.replace(r"^\s*$", np.nan, regex=True)

# Keep only the columns we need
KEEP_COLS = [
    # identifiers
    "sid", "name", "season",
    # basin
    "basin", "full_basin_name", "subbasin",
    # time & position
    "iso_time", "lat", "lon",
    # storm classification
    "nature", "track_type", "iflag", "usa_agency",
    # intensity
    "usa_wind", "usa_sshs",
    # derived / context
    "dist2land", "storm_speed",
]
df_ibtracs = df_ibtracs[KEEP_COLS].copy().sort_values(["sid", "iso_time"]).reset_index(drop=True)

print(
    f"after filter: {df_ibtracs.shape}  storms={df_ibtracs.sid.nunique():,}  "
    f"seasons {int(df_ibtracs.season.min())}–{int(df_ibtracs.season.max())}"
)

after filter: (97168, 17)  storms=4,254  seasons 1980–2025


/tmp/ipykernel_1287/3204968223.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_ibtracs = df_ibtracs.replace(r"^\s*$", np.nan, regex=True)


In [51]:
# @title Add past wind intensity trends

def offset_column_name(col, hours):
    '''hours < 0 is the past: ('usa_wind', -6) -> 'usa_wind_tminus6h'.'''
    return f"{col}_t{'plus' if hours > 0 else 'minus'}{abs(hours)}h"


def add_value_at_offset(df, col, hours, storm="sid", time="iso_time"):
    '''Add the value of `col` from `hours` hours away in the same storm.
    Negative looks back, positive looks forward. NaN when that storm has no
    observation at exactly that time. Does not mutate input, preserves row order.'''
    if not is_datetime64_any_dtype(df[time]):
        raise TypeError(f"{time!r} is {df[time].dtype}, expected datetime64. Run pd.to_datetime first.")
    offset_col = offset_column_name(col, hours)
    target_time = "_target_time"
    out = df.drop(columns=[offset_col, target_time], errors="ignore")
    repeats = out.duplicated([storm, time])
    if repeats.any():
        raise ValueError(f"{int(repeats.sum())} rows repeat a ({storm}, {time}) pair. Check the track_type filter.")
    lookup = out[[storm, time, col]].rename(columns={time: target_time, col: offset_col})
    out[target_time] = out[time] + pd.Timedelta(hours=hours)
    return out.merge(lookup, on=[storm, target_time], how="left", validate="one_to_one").drop(columns=[target_time])


def add_change_over_past_hours(df, col="usa_wind", lags=(6, 12, 24)):
    '''Add the change in `col` over each of the previous `lags` hours.'''
    out = df.copy()
    for hours in lags:
        out = add_value_at_offset(out, col, -hours)
        out[f"{col}_delta_tminus{hours}h"] = out[col] - out[offset_column_name(col, -hours)]
    return out

df_ibtracs = add_change_over_past_hours(df_ibtracs, "usa_wind", lags=(6, 12, 24))

In [52]:
# @title Defining the RI label

TARGET = f"ri_tplus{RI_HOURS_IN_FUTURE}"
def add_ri_label(df, col="usa_wind", hours=RI_HOURS_IN_FUTURE, threshold_kt=RI_THRESHOLD_KT):
    '''Add the forward change in `col` and the binary RI label. NA where the future value is unknown.'''
    delta_col = f"{col}_delta_tplus{hours}h"
    label_col = TARGET
    out = add_value_at_offset(df, col, hours).drop(columns=[delta_col, label_col], errors="ignore")
    out[delta_col] = out[offset_column_name(col, hours)] - out[col]
    out[label_col] = (out[delta_col] >= threshold_kt).astype("Int64").where(out[delta_col].notna())
    return out

df_ibtracs = add_ri_label(df_ibtracs, "usa_wind")
print(df_ibtracs[TARGET].value_counts(dropna=False))

n_before = len(df_ibtracs)
df_ibtracs = df_ibtracs[df_ibtracs[TARGET].notna()].copy().reset_index(drop=True)
n_lost = n_before - len(df_ibtracs)

print(f"Rows: {n_before:,} → {len(df_ibtracs):,}")
print(f"Rows removed: {n_lost:,}")
print(f"Unique tropical storms in the final dataframe: {df_ibtracs['sid'].nunique():,}")

ri_tplus24
0       72295
<NA>    18696
1        6177
Name: count, dtype: Int64
Rows: 97,168 → 78,472
Rows removed: 18,696
Unique tropical storms in the final dataframe: 4,105


In [53]:
df_ibtracs.columns

Index(['sid', 'name', 'season', 'basin', 'full_basin_name', 'subbasin',
       'iso_time', 'lat', 'lon', 'nature', 'track_type', 'iflag', 'usa_agency',
       'usa_wind', 'usa_sshs', 'dist2land', 'storm_speed', 'usa_wind_tminus6h',
       'usa_wind_delta_tminus6h', 'usa_wind_tminus12h',
       'usa_wind_delta_tminus12h', 'usa_wind_tminus24h',
       'usa_wind_delta_tminus24h', 'usa_wind_tplus24h',
       'usa_wind_delta_tplus24h', 'ri_tplus24'],
      dtype='object')

In [54]:
# @title Read ERA5 features from Parquet file
df_era5 = pd.read_parquet(ERA5_FEATURES_URL)
df_era5.head()
print(len(df_era5))

78472


In [55]:
# @title Read satellite features from Parquet file
df_satellite = pd.read_parquet(SATELLITE_FEATURES_URL)
df_satellite.head()
print(len(df_satellite))

78472


In [56]:
# @title Merge data from IBTrACS, ERA5 features, satellite features
df_era5 = df_era5.drop(columns=["lat", "lon"], errors="ignore")
df_satellite = df_satellite.drop(columns=["lat", "lon"], errors="ignore")

df_all_feats = (df_ibtracs
         .merge(df_era5, on=["sid", "iso_time"], how="left",
                validate="one_to_one", suffixes=("", "_DUP"))
         .merge(df_satellite, on=["sid", "iso_time"], how="left",
                validate="one_to_one", suffixes=("", "_DUP")))

dupes = [c for c in df_all_feats.columns if c.endswith("_DUP")]
assert not dupes, f"column collision in merge: {dupes}"

df_all_feats = df_all_feats[[c for c in df_all_feats.columns if c != TARGET] + [TARGET]]

In [ ]:
# @title Save the training seasons (1980-2015) for the XGBoost baseline (notebook 10)
# Same filters, features and label as the 2016-2025 file; no storm overlaps the development or test sets.
df_all_feats[df_all_feats["season"] <= 2015].to_csv(ROOT / "data" / "ibtracs_era5_sat_1980_2015.csv.gz", index=False, compression={"method": "gzip", "mtime": 0})

In [57]:
# @title Keep the 2016-2025 seasons (the development and test storms are sampled from these)
df_all_feats = df_all_feats[(df_all_feats['season'] >= 2016) & (df_all_feats['season'] < 2026)].copy()

# Absolute latitude, used only by the storm-identification probe's with-location version (notebook 05).
df_all_feats.insert(df_all_feats.columns.get_loc("usa_wind_delta_tminus24h") + 1, "abs_lat", df_all_feats["lat"].abs())

In [63]:
df_all_feats.to_csv(ROOT / "data" / "ibtracs_era5_sat_2016_2025.csv.gz", index=False, compression={"method": "gzip", "mtime": 0})